
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Tool Integration and Observability

## Overview
This lecture covers how deployed agents connect to external tools through the Model Context Protocol (MCP) and how MLflow Tracing provides the observability foundation for production monitoring. Together, these capabilities make agents extensible and observable — the prerequisites for evaluation and monitoring covered later in the course.

## Learning Objectives
By the end of this lecture you will be able to:
- Describe how MCP enables runtime tool discovery and invocation for deployed agents
- Differentiate managed, external, and custom MCP servers on Databricks
- Explain the MCP connection lifecycle from initialization through shutdown
- Identify what MLflow Tracing captures and how it connects deployment to evaluation

## A. Model Context Protocol

### A1. What Is MCP?

The **Model Context Protocol (MCP)** is an open protocol for connecting AI applications to external systems. Think of it like a USB-C port for AI: just as USB-C provides a standardized physical interface for connecting peripherals, MCP provides a standardized software interface that lets AI applications communicate with data sources, tools, and workflows.

Without MCP, every tool integration requires custom code: API wrappers, authentication handling, response parsing. MCP eliminates this by defining a common protocol that any compliant client and server can speak.

MCP follows a **client-server architecture** with three participants:

<div style="display:flex;align-items:center;justify-content:center;gap:0;margin:18px 0;font-family:system-ui,sans-serif;">
  <div style="border:2px solid #2574B5;border-radius:10px;padding:12px 14px;background:#fff;">
    <div style="font-size:14px;font-weight:600;color:#2574B5;letter-spacing:0.5px;text-transform:uppercase;margin-bottom:8px;">MCP Host (Your App)</div>
    <div style="background:#2574B5;color:#fff;padding:8px 14px;border-radius:6px;font-weight:600;font-size:14px;text-align:center;">MCP Client</div>
  </div>
  <div style="display:flex;flex-direction:column;align-items:center;padding:0 8px;">
    <div style="color:#02A36F;font-size:14px;font-weight:600;">tools/list</div>
    <div style="color:#999;font-size:20px;">&#x2194;</div>
    <div style="color:#FF5F46;font-size:14px;font-weight:600;">tools/call</div>
  </div>
  <div style="background:#1B5162;color:#fff;padding:14px 18px;border-radius:10px;font-weight:600;font-size:14px;text-align:center;min-width:110px;">MCP Server<br/><span style="font-weight:400;font-size:14px;">Tools, Resources, Prompts</span></div>
</div>

| Participant | Role |
|---|---|
| **MCP Host** | The AI application (e.g., a Databricks App) that manages one or more MCP clients |
| **MCP Client** | A component inside the host that maintains a dedicated connection to a single MCP server |
| **MCP Server** | A program that exposes tools, resources, or prompts to connected clients |

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">MCP servers expose three types of primitives: <strong>Tools</strong> (executable functions), <strong>Resources</strong> (read-only data sources), and <strong>Prompts</strong> (reusable interaction templates). In practice, tool primitives are the most common integration point for agents.</p>
</div>
</div>
</div>

Model Context Protocol overview ([AWS](https://docs.databricks.com/aws/en/generative-ai/mcp/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/mcp/) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/mcp/))

### A2. MCP on Databricks

Databricks provides three categories of MCP server to cover different integration needs:

| Type | Description | Example |
|---|---|---|
| **Managed MCP** | Pre-configured servers that give immediate access to Databricks features | Unity Catalog functions, Genie spaces, AI Search indexes |
| **External MCP** | Managed connections to MCP servers hosted outside of Databricks | Third-party APIs, SaaS integrations |
| **Custom MCP** | A custom MCP server you write and host as a Databricks App | Domain-specific tools, internal services |

The most common starting point is a **Managed MCP server backed by Unity Catalog functions**. You write a Python function, register it in UC, and the managed MCP server automatically exposes it as a tool that any connected agent can discover and invoke. The agent does not need custom code to call the function. The MCP protocol handles discovery, invocation, and response formatting.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">A single host can connect to multiple servers simultaneously, each through its own client instance. This lets an agent access tools from Unity Catalog functions, external APIs, and custom services all through the same protocol.</p>
</div>
</div>
</div>

Managed MCP servers ([AWS](https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/mcp/managed-mcp) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/mcp/managed-mcp))

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="https://files.training.databricks.com/binder/prod_main/deploying-and-monitoring-agents-on-databricks-en_us-1.0.0/images/20261022T103715Z/Deploying and Monitoring Agents on Databricks/Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to learn more about <strong>MCP server types on Databricks</strong>? Ask Genie Code. Click on the genie icon <img src="https://files.training.databricks.com/binder/prod_main/deploying-and-monitoring-agents-on-databricks-en_us-1.0.0/images/20261022T103715Z/Deploying and Monitoring Agents on Databricks/Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">What managed MCP servers are available on Databricks? How do I connect a Genie space as an MCP server in my agent?</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

### A3. MCP Lifecycle

<style>
.cb{margin-top:12px;border-radius:8px;overflow:hidden;border:0.5px solid #dde0e8}
.cb pre{background:#f7f8fc;padding:12px 14px;font-family:monospace;font-size:16px;line-height:1.7;overflow-x:auto;color:#0b2026}
.kw{color:#a855f7}.fn{color:#3b82f6}.st{color:#16a34a}.nm{color:#ea580c}.cm{color:#9ca3af}.dc{color:#d97706}
</style>

Every MCP connection follows a structured lifecycle:

<div style="display:flex;align-items:center;justify-content:center;gap:0;margin:18px 0;flex-wrap:wrap;font-family:system-ui,sans-serif;">
  <div style="background:#2574B5;color:#fff;padding:10px 16px;border-radius:8px;font-weight:600;font-size:15px;text-align:center;min-width:110px;">Initialize<br/><span style="font-weight:400;font-size:14px;">Capability handshake</span></div>
  <div style="color:#999;font-size:20px;padding:0 6px;">&#x2192;</div>
  <div style="border:2px dashed #999;border-radius:10px;padding:8px 10px;display:flex;align-items:center;gap:0;">
    <div style="position:relative;top:-20px;left:4px;font-size:14px;font-weight:600;color:#999;letter-spacing:0.5px;text-transform:uppercase;position:absolute;margin-top:-18px;">Operation</div>
    <div style="background:#02A36F;color:#fff;padding:10px 16px;border-radius:8px;font-weight:600;font-size:15px;text-align:center;min-width:110px;">Discover<br/><span style="font-weight:400;font-size:14px;">tools/list</span></div>
    <div style="color:#999;font-size:20px;padding:0 6px;">&#x2192;</div>
    <div style="background:#FF5F46;color:#fff;padding:10px 16px;border-radius:8px;font-weight:600;font-size:15px;text-align:center;min-width:110px;">Invoke<br/><span style="font-weight:400;font-size:14px;">tools/call</span></div>
  </div>
  <div style="color:#999;font-size:20px;padding:0 6px;">&#x2192;</div>
  <div style="background:#1B5162;color:#fff;padding:10px 16px;border-radius:8px;font-weight:600;font-size:15px;text-align:center;min-width:110px;">Shutdown<br/><span style="font-weight:400;font-size:14px;">Close connection</span></div>
</div>

- **Initialization**: The client sends an `initialize` request declaring its protocol version and capabilities. The server responds with its own capabilities. The client then sends an `initialized` notification to confirm readiness before normal message exchange begins.
- **Tool discovery**: The client sends a `tools/list` request. The server returns all available tools with their names, descriptions, and input schemas. The agent uses these descriptions to decide when to call each tool.
- **Tool invocation**: When the agent decides to use a tool, the client sends a `tools/call` request with the tool name and arguments. The server executes the function and returns the result.
- **Shutdown**: Either side can close the connection gracefully.

<details style="margin-top:12px;">
<summary style="cursor:pointer;font-weight:600;font-size:15px;color:#378ADD;">Example: Connecting to UC functions via MCP</summary>
<div class="cb"><pre><span class="cm"># MCP integration using the Databricks OpenAI connector</span>
<span class="kw">from</span> databricks.sdk <span class="kw">import</span> WorkspaceClient
<span class="kw">from</span> databricks_openai.agents <span class="kw">import</span> McpServer
<span class="kw">from</span> agents <span class="kw">import</span> Agent, Runner

workspace_client = <span class="fn">WorkspaceClient</span>()

<span class="cm"># 1. Initialize -- connect to UC functions as an MCP server (async context manager)</span>
<span class="kw">async with</span> <span class="fn">McpServer.from_uc_function</span>(
    catalog=<span class="st">"my_catalog"</span>,
    schema=<span class="st">"my_schema"</span>,
    workspace_client=workspace_client,
    name=<span class="st">"uc-functions"</span>,
) <span class="kw">as</span> uc_server:
    <span class="cm"># 2. Discovery -- tools are listed automatically</span>
    <span class="cm"># 3. Invocation -- agent calls tools as needed</span>
    agent = <span class="fn">Agent</span>(
        name=<span class="st">"my_agent"</span>,
        model=<span class="st">"databricks-claude-sonnet-4-5"</span>,
        mcp_servers=[uc_server],
    )
    result = <span class="kw">await</span> <span class="fn">Runner.run</span>(agent, <span class="st">"Look up info for Acme Corp"</span>)</pre></div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Tools are <strong>discovered at runtime</strong>, not hardcoded. If a new UC function is registered on the MCP server, connected agents can discover and use it without redeployment. When MLflow Tracing is active, every MCP tool call is captured as a span within the trace.</p>
</div>
</div>
</div>


### A4. MCP in a Deployed App

<style>
.cb{margin-top:12px;border-radius:8px;overflow:hidden;border:0.5px solid #dde0e8}
.cb pre{background:#f7f8fc;padding:12px 14px;font-family:monospace;font-size:16px;line-height:1.7;overflow-x:auto;color:#0b2026}
.kw{color:#a855f7}.fn{color:#3b82f6}.st{color:#16a34a}.nm{color:#ea580c}.cm{color:#9ca3af}.dc{color:#d97706}
</style>

In a Databricks App, MCP servers are typically initialized at startup and shared across requests. The app's service principal must have permissions to access the underlying resources (UC functions, external endpoints) that the MCP servers expose.

<details style="margin-top:12px;">
<summary style="cursor:pointer;font-weight:600;font-size:15px;color:#378ADD;">Example: FastAPI server with MCP tool integration</summary>
<div class="cb"><pre><span class="cm"># server.py -- FastAPI app with MCP integration</span>
<span class="kw">from</span> contextlib <span class="kw">import</span> asynccontextmanager
<span class="kw">from</span> fastapi <span class="kw">import</span> FastAPI
<span class="kw">from</span> databricks.sdk <span class="kw">import</span> WorkspaceClient
<span class="kw">from</span> databricks_openai.agents <span class="kw">import</span> McpServer
<span class="kw">from</span> agents <span class="kw">import</span> Agent, Runner
<span class="kw">import</span> os

workspace_client = <span class="fn">WorkspaceClient</span>()

<span class="dc">@asynccontextmanager</span>
<span class="kw">async def</span> <span class="fn">lifespan</span>(app):
    <span class="cm"># Initialize MCP server at startup (async context manager)</span>
    <span class="kw">async with</span> <span class="fn">McpServer.from_uc_function</span>(
        catalog=os.environ[<span class="st">"UC_TOOL_CATALOG"</span>],
        schema=os.environ[<span class="st">"UC_TOOL_SCHEMA"</span>],
        workspace_client=workspace_client,
        name=<span class="st">"uc-functions"</span>,
    ) <span class="kw">as</span> uc_server:
        app.state.agent = <span class="fn">Agent</span>(
            name=<span class="st">"deployed_agent"</span>,
            model=os.environ[<span class="st">"SERVING_ENDPOINT"</span>],
            mcp_servers=[uc_server],
        )
        <span class="kw">yield</span>

app = <span class="fn">FastAPI</span>(lifespan=lifespan)

<span class="dc">@app.post</span>(<span class="st">"/chat"</span>)
<span class="kw">async def</span> <span class="fn">chat</span>(request: ChatRequest):
    result = <span class="kw">await</span> <span class="fn">Runner.run</span>(app.state.agent, request.message)
    <span class="kw">return</span> {<span class="st">"response"</span>: result.final_output}</pre></div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">The UC catalog and schema are injected as environment variables through <code>databricks.yml</code>, keeping the app code portable across environments. The service principal assigned to the app must have <code>EXECUTE</code> permission on the UC functions. The <code>McpServer.from_uc_function</code> must be used as an async context manager to properly manage the server connection lifecycle.</p>
</div>
</div>
</div>


## B. Observability with MLflow Tracing

### B1. Adding Tracing to Your Agent

<style>
.cb{margin-top:12px;border-radius:8px;overflow:hidden;border:0.5px solid #dde0e8}
.cb pre{background:#f7f8fc;padding:12px 14px;font-family:monospace;font-size:16px;line-height:1.7;overflow-x:auto;color:#0b2026}
.kw{color:#a855f7}.fn{color:#3b82f6}.st{color:#16a34a}.nm{color:#ea580c}.cm{color:#9ca3af}.dc{color:#d97706}
</style>

MLflow Tracing records the execution of each agent request as a structured **trace**, a tree of **spans** that represent individual operations. The simplest way to enable tracing is the `@mlflow.trace` decorator on your agent's entry point.

Each span captures:
- **Inputs and outputs**: What went into the operation and what came out
- **Timing**: Start time, end time, and duration
- **Span type**: The kind of operation (e.g., `AGENT`, `CHAT_MODEL`, `TOOL`, `RETRIEVER`)
- **Status**: Whether the operation succeeded or failed

If the agent calls supported libraries (e.g., the OpenAI SDK), child spans are generated automatically for each library call. You can also add manual spans with `mlflow.start_span()` for custom logic.

<div style="margin:18px auto;font-family:'Menlo','Consolas',monospace;font-size:14px;line-height:2;background:#f7f8fc;border:1px solid #dde0e8;border-radius:10px;padding:18px 22px;max-width:480px;">
  <div style="color:#2574B5;font-weight:700;">AGENT <span style="color:#999;font-weight:400;">handle_request</span></div>
  <div style="padding-left:20px;border-left:2px solid #dde0e8;margin-left:8px;">
    <div style="color:#F8A805;font-weight:700;">CHAT_MODEL <span style="color:#999;font-weight:400;">chat.completions</span></div>
    <div style="color:#02A36F;font-weight:700;">TOOL <span style="color:#999;font-weight:400;">lookup_weather</span></div>
    <div style="color:#F8A805;font-weight:700;">CHAT_MODEL <span style="color:#999;font-weight:400;">chat.completions</span></div>
  </div>
</div>

<details style="margin-top:12px;">
<summary style="cursor:pointer;font-weight:600;font-size:15px;color:#378ADD;">Example: @mlflow.trace decorator and manual spans</summary>
<div class="cb"><pre><span class="kw">import</span> mlflow

<span class="dc">@mlflow.trace</span>
<span class="kw">def</span> <span class="fn">handle_request</span>(user_message: <span class="fn">str</span>) -> <span class="fn">str</span>:
    <span class="cm"># Child spans generated automatically for OpenAI calls</span>
    <span class="cm"># CHAT_MODEL spans capture LLM interactions</span>
    <span class="cm"># TOOL spans capture MCP tool invocations</span>
    response = agent.<span class="fn">run</span>(user_message)
    <span class="kw">return</span> response

<span class="cm"># Manual span for custom logic</span>
<span class="dc">@mlflow.trace</span>
<span class="kw">def</span> <span class="fn">handle_request_v2</span>(user_message: <span class="fn">str</span>) -> <span class="fn">str</span>:
    <span class="kw">with</span> mlflow.<span class="fn">start_span</span>(name=<span class="st">"preprocessing"</span>) <span class="kw">as</span> span:
        cleaned = <span class="fn">preprocess</span>(user_message)
        span.<span class="fn">set_inputs</span>({<span class="st">"raw"</span>: user_message})
        span.<span class="fn">set_outputs</span>({<span class="st">"cleaned"</span>: cleaned})
    response = agent.<span class="fn">run</span>(cleaned)
    <span class="kw">return</span> response</pre></div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">A typical agent trace contains a root <code>AGENT</code> span with child spans for <code>CHAT_MODEL</code> calls, <code>TOOL</code> invocations, and <code>RETRIEVER</code> operations. This hierarchy shows exactly how the agent processed the request: which tools it called, in what order, and how long each step took.</p>
</div>
</div>
</div>

MLflow Tracing with function decorators ([AWS](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/app-instrumentation/manual-tracing/function-decorator) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/app-instrumentation/manual-tracing/function-decorator) | [GCP](https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/app-instrumentation/manual-tracing/function-decorator))

### B2. Traces in Production

<style>
.cb{margin-top:12px;border-radius:8px;overflow:hidden;border:0.5px solid #dde0e8}
.cb pre{background:#f7f8fc;padding:12px 14px;font-family:monospace;font-size:16px;line-height:1.7;overflow-x:auto;color:#0b2026}
.kw{color:#a855f7}.fn{color:#3b82f6}.st{color:#16a34a}.nm{color:#ea580c}.cm{color:#9ca3af}.dc{color:#d97706}
</style>

In production, traces are logged to an **MLflow Experiment** backed by Unity Catalog. This means traces are stored as governed objects with full lineage and access control. The experiment serves as the central hub where all observability and evaluation data converges.

Traces are the foundation that the rest of the monitoring pipeline builds on:
- **Scorers evaluate traces**, not raw requests. Any agent that produces MLflow traces can be evaluated using the same scorer infrastructure.
- **Trace archival** streams traces to Delta tables for SQL-based analysis and dashboarding.
- **Metric backfill** applies new scorers retroactively to traces logged before the scorer existed.

The experiment name is typically injected as an environment variable through `databricks.yml`, so the agent code sets it at startup:

<details style="margin-top:12px;">
<summary style="cursor:pointer;font-weight:600;font-size:15px;color:#378ADD;">Example: Setting the MLflow experiment at startup</summary>
<div class="cb"><pre><span class="kw">import</span> mlflow
<span class="kw">import</span> os

mlflow.<span class="fn">set_experiment</span>(os.environ[<span class="st">"MLFLOW_EXPERIMENT_NAME"</span>])</pre></div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Tracing must be enabled before scorers can be useful. If your agent is not producing traces, there is nothing for scorers to evaluate. The first step in any monitoring setup is ensuring traces are flowing to an MLflow Experiment.</p>
</div>
</div>
</div>

MLflow Tracing overview ([AWS](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/) | [GCP](https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/))

### B3. Storing Traces in Unity Catalog

<style>
.cb{margin-top:12px;border-radius:8px;overflow:hidden;border:0.5px solid #dde0e8}
.cb pre{background:#f7f8fc;padding:12px 14px;font-family:monospace;font-size:16px;line-height:1.7;overflow-x:auto;color:#0b2026}
.kw{color:#a855f7}.fn{color:#3b82f6}.st{color:#16a34a}.nm{color:#ea580c}.cm{color:#9ca3af}.dc{color:#d97706}
</style>

By default, traces are stored in the MLflow control plane. For production workloads, you can store OpenTelemetry traces directly in **Unity Catalog Delta tables**. This gives you long-term retention, SQL-queryable trace data, and governance through UC schema and table permissions rather than experiment-level ACLs.

To bind an experiment to Unity Catalog storage, pass a `trace_location` parameter when creating or setting the experiment:

<details style="margin-top:12px;">
<summary style="cursor:pointer;font-weight:600;font-size:15px;color:#378ADD;">Example: Creating a UC-backed experiment</summary>
<div class="cb"><pre><span class="kw">from</span> mlflow.entities.trace_location <span class="kw">import</span> UnityCatalog

experiment = mlflow.<span class="fn">set_experiment</span>(
    experiment_name=<span class="st">"my-agent-experiment"</span>,
    trace_location=<span class="fn">UnityCatalog</span>(
        catalog_name=<span class="st">"my_catalog"</span>,
        schema_name=<span class="st">"my_schema"</span>,
        table_prefix=<span class="st">"agent_traces"</span>,
    ),
)</pre></div>
</details>

This automatically creates four Delta tables in the specified schema:

| Table | Contents |
|---|---|
| `<prefix>_otel_spans` | Individual span records (the primary trace data) |
| `<prefix>_otel_annotations` | Assessments and feedback attached to traces |
| `<prefix>_otel_logs` | Log entries associated with trace execution |
| `<prefix>_otel_metrics` | Metric values computed by scorers |

Once stored in Delta tables, traces can be queried directly with SQL through a Databricks SQL warehouse, enabling custom reporting and dashboards.

**Governance**: Access is controlled through UC table permissions. Users need `USE_CATALOG`, `USE_SCHEMA`, and explicit `MODIFY` + `SELECT` grants on each table. Note that `ALL_PRIVILEGES` is **not sufficient** — `MODIFY` and `SELECT` must be granted explicitly.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Once you bind an experiment to a UC trace location, you <strong>cannot reassign</strong> it to a different location. However, multiple experiments can share the same UC location. Users with access to the UC tables can view all traces stored there, regardless of which experiment the traces belong to. Ingestion is capped at 200 traces per second per workspace and 100 MB per second per table.</p>
</div>
</div>
</div>

Traces in Unity Catalog ([AWS](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/trace-unity-catalog) | [GCP](https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/trace-unity-catalog))

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
